
# PKC isoform expression across D0-D3 — plotnine version

This notebook calculates day-level pseudobulk expression for the canonical human PKC isoforms:

- conventional PKCs: `PRKCA`, `PRKCB`, `PRKCG`;
- novel PKCs: `PRKCD`, `PRKCE`, `PRKCH`, `PRKCQ`;
- atypical PKCs: `PRKCI`, `PRKCZ`.

The primary output is **pseudobulk TPM** calculated from raw UMI counts and gene lengths. The notebook also reports UMI counts and pseudobulk counts per million (CPM).

The newly supplied program-annotation notebook does not itself contain an AnnData path. The integrated AnnData path below is the path consistently used in the previously supplied CC-Perturb-seq UMAP and Hippo notebooks.


In [ ]:

from pathlib import Path
from collections import OrderedDict, defaultdict
import gzip
import re
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from plotnine import (
    aes,
    element_blank,
    element_rect,
    element_text,
    facet_grid,
    geom_line,
    geom_point,
    geom_text,
    geom_tile,
    ggplot,
    labs,
    scale_color_brewer,
    scale_fill_cmap,
    scale_x_discrete,
    theme,
    theme_classic,
    theme_minimal,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

class cfg:
    ADATA_H5AD = Path(
        "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
        "2.preprocessing_tonys_pipeline/3.integrated_days/"
        "3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad"
    )

    DAY_OBS_COL = "day"
    OUTPUT_DIR = Path("PKC_isoform_expression_D0_D3_outputs")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # True TPM requires gene lengths. This notebook uses the GENCODE v43
    # annotation GTF supplied on Sherlock when a suitable length column is absent.
    GENCODE_V43_GTF = Path("/oak/stanford/groups/engreitz/Users/tri/tmp/gencode.v43.annotation.gtf.gz")

    COUNT_LAYER_CANDIDATES = ["counts", "raw_counts", "count"]
    GENE_SYMBOL_COL_CANDIDATES = [
        "gene_symbol", "gene_name", "symbol", "feature_name"
    ]
    GENE_ID_COL_CANDIDATES = [
        "gene_id", "ensembl_id", "feature_id"
    ]
    GENE_LENGTH_COL_CANDIDATES = [
        "gene_length_bp", "gene_length", "feature_length", "length"
    ]

    CELL_CHUNK_SIZE = 2000

    PKC_GROUPS = OrderedDict(
        [
            ("Conventional PKC", ["PRKCA", "PRKCB", "PRKCG"]),
            ("Novel PKC", ["PRKCD", "PRKCE", "PRKCH", "PRKCQ"]),
            ("Atypical PKC", ["PRKCI", "PRKCZ"]),
        ]
    )

    # These are protein kinase D genes, not PKC isoforms. Set True only
    # when a separate related-kinase comparison is desired.
    INCLUDE_PKD_FAMILY = False
    PKD_GENES = ["PRKD1", "PRKD2", "PRKD3"]

PKC_GENES = [
    gene
    for genes in cfg.PKC_GROUPS.values()
    for gene in genes
]
if cfg.INCLUDE_PKD_FAMILY:
    PKC_GENES += cfg.PKD_GENES

mpl.rcParams.update(
    {
        # Preserve all plotnine labels as native text in SVG and editable
        # TrueType text in PDF for Adobe Illustrator.
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "text.usetex": False,
        "font.family": "Liberation Sans",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    }
)

print("AnnData:", cfg.ADATA_H5AD)
print("Genes:", PKC_GENES)



## 1. Load the integrated AnnData and validate the day labels


In [ ]:

if not cfg.ADATA_H5AD.exists():
    raise FileNotFoundError(
        f"AnnData not found: {cfg.ADATA_H5AD}\n"
        "Run this notebook on Sherlock with /oak mounted, or update cfg.ADATA_H5AD."
    )

adata = sc.read_h5ad(cfg.ADATA_H5AD, backed="r")
print(adata)
print(f"Cells: {adata.n_obs:,}")
print(f"Features: {adata.n_vars:,}")

if cfg.DAY_OBS_COL not in adata.obs.columns:
    raise KeyError(
        f"{cfg.DAY_OBS_COL!r} is not in adata.obs. "
        f"Available columns include: {list(adata.obs.columns[:50])}"
    )

def normalize_day(value):
    text = str(value).strip().upper().replace("DAY", "D")
    match = re.search(r"([0-3])", text)
    return f"D{match.group(1)}" if match else None

day_labels = adata.obs[cfg.DAY_OBS_COL].map(normalize_day)
day_counts = day_labels.value_counts(dropna=False).sort_index()
display(day_counts.rename("cells").to_frame())

missing_days = {"D0", "D1", "D2", "D3"} - set(day_labels.dropna())
if missing_days:
    raise RuntimeError(
        "Missing expected day labels: " + ", ".join(sorted(missing_days))
    )



## 2. Resolve a raw-count matrix and feature metadata

The notebook will not calculate TPM from log-normalized expression. It prefers an explicit raw-count layer, then `adata.raw`, and uses `adata.X` only if sampled values are count-like non-negative integers.


In [ ]:

def first_existing_column(frame, candidates):
    lowered = {str(c).lower(): c for c in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    return None


def nonzero_values(matrix):
    if sparse.issparse(matrix):
        return np.asarray(matrix.data)
    array = np.asarray(matrix)
    return array[np.isfinite(array) & (array != 0)]


def matrix_is_count_like(accessor, n_obs):
    stop = min(n_obs, 1000)
    sample = accessor(0, stop)
    values = nonzero_values(sample)
    if values.size == 0:
        return True
    values = values[: min(values.size, 100000)]
    return (
        np.nanmin(values) >= 0
        and np.mean(np.isclose(values, np.round(values), atol=1e-6)) >= 0.995
    )


count_source = None
matrix_accessor = None
feature_frame = None
feature_names = None

for layer in cfg.COUNT_LAYER_CANDIDATES:
    if layer in adata.layers:
        count_source = f"adata.layers[{layer!r}]"
        matrix_accessor = lambda start, stop, key=layer: adata.layers[key][start:stop, :]
        feature_frame = adata.var.copy()
        feature_names = pd.Index(adata.var_names.astype(str))
        break

if matrix_accessor is None and adata.raw is not None:
    count_source = "adata.raw.X"
    matrix_accessor = lambda start, stop: adata.raw.X[start:stop, :]
    feature_frame = adata.raw.var.copy()
    feature_names = pd.Index(adata.raw.var_names.astype(str))

if matrix_accessor is None:
    candidate_accessor = lambda start, stop: adata.X[start:stop, :]
    if matrix_is_count_like(candidate_accessor, adata.n_obs):
        count_source = "adata.X (verified count-like)"
        matrix_accessor = candidate_accessor
        feature_frame = adata.var.copy()
        feature_names = pd.Index(adata.var_names.astype(str))
    else:
        raise RuntimeError(
            "No trusted raw-count source was found. TPM cannot be calculated "
            "from log-normalized adata.X. Add a raw-count layer or use an h5ad "
            "that preserves counts."
        )

print("Count source:", count_source)
print("Count matrix shape:", (adata.n_obs, len(feature_names)))
display(feature_frame.head())



## 3. Resolve gene symbols, Ensembl IDs, and gene lengths

When no suitable length column exists in `adata.var`/`adata.raw.var`, set `cfg.GENCODE_V43_GTF` to the absolute path of `gencode.v43.annotation.gtf` or `.gtf.gz`. The parser calculates union-exon length per gene.


In [ ]:

symbol_col = first_existing_column(
    feature_frame,
    cfg.GENE_SYMBOL_COL_CANDIDATES,
)
gene_id_col = first_existing_column(
    feature_frame,
    cfg.GENE_ID_COL_CANDIDATES,
)
length_col = first_existing_column(
    feature_frame,
    cfg.GENE_LENGTH_COL_CANDIDATES,
)

gene_symbols = (
    feature_frame[symbol_col].astype(str).to_numpy()
    if symbol_col is not None
    else feature_names.astype(str).to_numpy()
)
gene_ids = (
    feature_frame[gene_id_col].astype(str).str.replace(r"\.\d+$", "", regex=True).to_numpy()
    if gene_id_col is not None
    else feature_names.astype(str).str.replace(r"\.\d+$", "", regex=True).to_numpy()
)

print("Gene-symbol source:", symbol_col or "feature names")
print("Gene-ID source:", gene_id_col or "feature names")
print("Gene-length column:", length_col)


def parse_gtf_attributes(text):
    result = {}
    for field in text.strip().strip(";").split(";"):
        field = field.strip()
        if not field:
            continue
        if " " not in field:
            continue
        key, value = field.split(" ", 1)
        result[key] = value.strip().strip('"')
    return result


def merge_interval_length(intervals):
    if not intervals:
        return 0
    intervals = sorted(intervals)
    total = 0
    current_start, current_end = intervals[0]
    for start, end in intervals[1:]:
        if start <= current_end + 1:
            current_end = max(current_end, end)
        else:
            total += current_end - current_start + 1
            current_start, current_end = start, end
    total += current_end - current_start + 1
    return total


def gene_lengths_from_gtf(gtf_path, wanted_ids, wanted_symbols):
    path = Path(gtf_path)
    if not path.exists():
        raise FileNotFoundError(path)

    wanted_ids = set(str(x).split(".")[0] for x in wanted_ids)
    wanted_symbols = set(map(str, wanted_symbols))
    exon_intervals = defaultdict(list)
    gene_alias = {}

    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if not line or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] != "exon":
                continue
            attrs = parse_gtf_attributes(fields[8])
            gene_id = attrs.get("gene_id", "").split(".")[0]
            gene_name = attrs.get("gene_name", "")
            if gene_id not in wanted_ids and gene_name not in wanted_symbols:
                continue
            key = gene_id if gene_id in wanted_ids else gene_name
            exon_intervals[key].append((int(fields[3]), int(fields[4])))
            gene_alias[key] = (gene_id, gene_name)

    by_id = {}
    by_symbol = {}
    for key, intervals in exon_intervals.items():
        length = merge_interval_length(intervals)
        gene_id, gene_name = gene_alias[key]
        if gene_id:
            by_id[gene_id] = length
        if gene_name:
            by_symbol[gene_name] = length
    return by_id, by_symbol


if length_col is not None:
    gene_lengths_bp = pd.to_numeric(
        feature_frame[length_col],
        errors="coerce",
    ).to_numpy()
    length_source = f"feature metadata column {length_col!r}"
else:
    if cfg.GENCODE_V43_GTF is None:
        raise RuntimeError(
            "No gene-length column was found. Set cfg.GENCODE_V43_GTF to "
            "the absolute Sherlock path of gencode.v43.annotation.gtf or .gtf.gz. "
            "The notebook will not label CPM as TPM."
        )

    lengths_by_id, lengths_by_symbol = gene_lengths_from_gtf(
        cfg.GENCODE_V43_GTF,
        gene_ids,
        gene_symbols,
    )
    gene_lengths_bp = np.array(
        [
            lengths_by_id.get(str(gid).split(".")[0],
                              lengths_by_symbol.get(str(symbol), np.nan))
            for gid, symbol in zip(gene_ids, gene_symbols)
        ],
        dtype=float,
    )
    length_source = f"union-exon lengths from {cfg.GENCODE_V43_GTF}"

print("Gene-length source:", length_source)
print(
    f"Features with valid positive lengths: "
    f"{np.sum(np.isfinite(gene_lengths_bp) & (gene_lengths_bp > 0)):,} / "
    f"{len(gene_lengths_bp):,}"
)

if np.sum(np.isfinite(gene_lengths_bp) & (gene_lengths_bp > 0)) < 1000:
    raise RuntimeError(
        "Too few valid gene lengths were resolved for a reliable TPM denominator."
    )



## 4. Sum raw UMI counts by day in chunks


In [ ]:

days = ["D0", "D1", "D2", "D3"]
day_to_index = {day: i for i, day in enumerate(days)}
pseudobulk_counts = np.zeros(
    (len(days), len(feature_names)),
    dtype=np.float64,
)

day_array = day_labels.to_numpy()

for start in range(0, adata.n_obs, cfg.CELL_CHUNK_SIZE):
    stop = min(start + cfg.CELL_CHUNK_SIZE, adata.n_obs)
    matrix = matrix_accessor(start, stop)
    chunk_days = day_array[start:stop]

    for day in days:
        local_rows = np.where(chunk_days == day)[0]
        if local_rows.size == 0:
            continue
        block = matrix[local_rows, :]
        summed = np.asarray(block.sum(axis=0)).ravel()
        pseudobulk_counts[day_to_index[day], :] += summed

    if start % (cfg.CELL_CHUNK_SIZE * 10) == 0:
        print(f"Processed {stop:,} / {adata.n_obs:,} cells")

print("Completed pseudobulk aggregation.")



## 5. Calculate pseudobulk CPM and TPM

For day \(d\) and gene \(g\):

\[
\mathrm{RPK}_{dg} = rac{\mathrm{UMI}_{dg}}{\mathrm{gene\ length}_{g}/1000},
\]

\[
\mathrm{TPM}_{dg} =
10^6 	imes
rac{\mathrm{RPK}_{dg}}
{\sum_h \mathrm{RPK}_{dh}}.
\]


In [ ]:

library_totals = pseudobulk_counts.sum(axis=1)
cpm = pseudobulk_counts / library_totals[:, None] * 1e6

valid_length = np.isfinite(gene_lengths_bp) & (gene_lengths_bp > 0)
rpk = np.zeros_like(pseudobulk_counts, dtype=float)
rpk[:, valid_length] = (
    pseudobulk_counts[:, valid_length]
    / (gene_lengths_bp[valid_length][None, :] / 1000.0)
)

rpk_totals = rpk.sum(axis=1)
if np.any(rpk_totals <= 0):
    raise RuntimeError("At least one day has a non-positive RPK total.")

tpm = rpk / rpk_totals[:, None] * 1e6

summary_rows = []
for family, genes in cfg.PKC_GROUPS.items():
    for gene in genes:
        feature_indices = np.where(gene_symbols == gene)[0]
        if feature_indices.size == 0:
            summary_rows.append(
                {
                    "PKC_family": family,
                    "gene": gene,
                    "status": "not found",
                }
            )
            continue

        if feature_indices.size > 1:
            print(
                f"WARNING: {gene} maps to {feature_indices.size} features; "
                "values are summed across matching features."
            )

        for day_index, day in enumerate(days):
            summary_rows.append(
                {
                    "PKC_family": family,
                    "gene": gene,
                    "day": day,
                    "status": "found",
                    "raw_pseudobulk_UMIs": float(
                        pseudobulk_counts[day_index, feature_indices].sum()
                    ),
                    "pseudobulk_CPM": float(
                        cpm[day_index, feature_indices].sum()
                    ),
                    "pseudobulk_TPM": float(
                        tpm[day_index, feature_indices].sum()
                    ),
                    "gene_length_bp": float(
                        np.nanmax(gene_lengths_bp[feature_indices])
                    ),
                    "n_matching_features": int(feature_indices.size),
                    "count_source": count_source,
                    "length_source": length_source,
                    "cells_in_day": int((day_array == day).sum()),
                }
            )

expression_long = pd.DataFrame(summary_rows)
display(expression_long)

expression_long.to_csv(
    cfg.OUTPUT_DIR / "PKC_isoform_expression_D0_D3_long.csv",
    index=False,
)

expression_wide = (
    expression_long.loc[expression_long["status"].eq("found")]
    .pivot(index=["PKC_family", "gene"], columns="day", values="pseudobulk_TPM")
    .reindex(columns=days)
    .reset_index()
)
expression_wide.to_csv(
    cfg.OUTPUT_DIR / "PKC_isoform_TPM_D0_D3_wide.csv",
    index=False,
)
display(expression_wide)



## 6. Plot PKC isoform TPM across D0-D3

Both a line plot and a heatmap are displayed and exported as PNG, PDF and Illustrator-editable SVG.


### Illustrator editability

SVG files are exported with `svg.fonttype = "none"` and no LaTeX/mathtext labels, so all plot labels remain native, editable text objects in Adobe Illustrator. PDF files use TrueType text (`pdf.fonttype = 42`).

In [ ]:
plot_long = expression_long.loc[
    expression_long["status"].eq("found")
].copy()

plot_long["day"] = pd.Categorical(
    plot_long["day"],
    categories=days,
    ordered=True,
)
plot_long["gene"] = pd.Categorical(
    plot_long["gene"],
    categories=PKC_GENES,
    ordered=True,
)
family_order = list(cfg.PKC_GROUPS.keys())
plot_long["PKC_family"] = pd.Categorical(
    plot_long["PKC_family"],
    categories=family_order,
    ordered=True,
)

line_plot = (
    ggplot(
        plot_long,
        aes(
            x="day",
            y="pseudobulk_TPM",
            group="gene",
            color="gene",
        ),
    )
    + geom_line(size=0.9, alpha=0.9)
    + geom_point(size=2.2, alpha=0.95)
    + facet_grid(
        "PKC_family ~ .",
        scales="free_y",
        space="free_y",
    )
    + scale_x_discrete(limits=days)
    + scale_color_brewer(type="qual", palette="Dark2")
    + labs(
        title="PKC isoform expression during hiPSC-to-EC differentiation",
        subtitle="Day-level pseudobulk expression from raw UMI counts",
        x="Differentiation day",
        y="Pseudobulk TPM",
        color=None,
    )
    + theme_classic(base_size=10.5, base_family="Liberation Sans")
    + theme(
        figure_size=(7.0, 6.8),
        text=element_text(family="Liberation Sans"),
        plot_title=element_text(size=14, weight="bold", ha="left"),
        plot_subtitle=element_text(size=9.5, color="#555555", ha="left"),
        axis_title_x=element_text(size=10.5),
        axis_title_y=element_text(size=10.5),
        axis_text_x=element_text(size=9.5),
        axis_text_y=element_text(size=9),
        strip_background=element_rect(
            fill="#F2F2F2",
            color="#D9D9D9",
            size=0.6,
        ),
        strip_text_y=element_text(
            size=9.5,
            weight="bold",
            angle=0,
        ),
        legend_position="right",
        legend_text=element_text(size=8.3),
        panel_spacing_y=0.10,
    )
)

display(line_plot)
for extension in ["png", "pdf", "svg"]:
    line_plot.save(
        cfg.OUTPUT_DIR / f"PKC_isoform_TPM_D0_D3_plotnine_lineplot_editable_text.{extension}",
        format=extension,
        dpi=350 if extension == "png" else 300,
        verbose=False,
    )

heatmap_long = plot_long[
    ["PKC_family", "gene", "day", "pseudobulk_TPM"]
].copy()
heatmap_long["label"] = heatmap_long["pseudobulk_TPM"].map(
    lambda value: f"{value:.1f}" if pd.notna(value) else ""
)

heatmap_plot = (
    ggplot(
        heatmap_long,
        aes(
            x="day",
            y="gene",
            fill="pseudobulk_TPM",
        ),
    )
    + geom_tile(color="white", size=0.5)
    + geom_text(
        aes(label="label"),
        size=8.0,
        color="#202020",
    )
    + facet_grid(
        "PKC_family ~ .",
        scales="free_y",
        space="free_y",
    )
    + scale_x_discrete(limits=days)
    + scale_fill_cmap(name="TPM", cmap_name="YlGnBu")
    + labs(
        title="PKC isoform expression across D0-D3",
        subtitle="Day-level pseudobulk TPM",
        x="Differentiation day",
        y=None,
    )
    + theme_minimal(base_size=10.5, base_family="Liberation Sans")
    + theme(
        figure_size=(5.4, 6.2),
        text=element_text(family="Liberation Sans"),
        plot_title=element_text(size=14, weight="bold", ha="left"),
        plot_subtitle=element_text(size=9.5, color="#555555", ha="left"),
        axis_title_x=element_text(size=10.5),
        axis_text_x=element_text(size=9.5),
        axis_text_y=element_text(size=10),
        panel_grid=element_blank(),
        strip_background=element_rect(
            fill="#F2F2F2",
            color="#D9D9D9",
            size=0.6,
        ),
        strip_text_y=element_text(
            size=9.5,
            weight="bold",
            angle=0,
        ),
        legend_position="right",
        panel_spacing_y=0.10,
    )
)

display(heatmap_plot)
for extension in ["png", "pdf", "svg"]:
    heatmap_plot.save(
        cfg.OUTPUT_DIR / f"PKC_isoform_TPM_D0_D3_plotnine_heatmap_editable_text.{extension}",
        format=extension,
        dpi=350 if extension == "png" else 300,
        verbose=False,
    )


# Confirm that Illustrator-editable SVG text objects were preserved.
for svg_name in [
    "PKC_isoform_TPM_D0_D3_plotnine_lineplot_editable_text.svg",
    "PKC_isoform_TPM_D0_D3_plotnine_heatmap_editable_text.svg",
]:
    svg_path = cfg.OUTPUT_DIR / svg_name
    svg_source = svg_path.read_text(encoding="utf-8")
    if "<text" not in svg_source:
        raise RuntimeError(f"No native text objects found in {svg_path}")
    print(f"{svg_name}: {svg_source.count('<text')} editable SVG text objects")



## 7. Close the backed AnnData handle


In [ ]:

try:
    adata.file.close()
except Exception:
    pass

print("Outputs:", cfg.OUTPUT_DIR.resolve())
